In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib

# ---------------------------
# 🔹 Load & Validate Data
# ---------------------------
try:
    data = pd.read_csv("new_dataset.csv")  # Ensure this file is the cleaned dataset
    print("✅ Dataset loaded successfully!")
except FileNotFoundError:
    print("❌ Error: File not found. Ensure 'new_dataset.csv' exists in the correct path.")
    exit()

# 🔹 Display dataset info
print("\n📊 Dataset Info:")
print(data.info())
print("\n🔍 First 5 Rows:")
print(data.head())

# ---------------------------
# 🔹 Data Preprocessing
# ---------------------------
# Drop the 'user_id' column since it's not needed for training
if 'user_id' in data.columns:
    data = data.drop(columns=['user_id'])

# Convert all columns to numeric values
data = data.apply(pd.to_numeric, errors='coerce')

# Drop any empty rows after conversion
data = data.dropna()

# Final dataset check
if data.shape[0] == 0:
    print("❌ Error: No data left after cleaning. Check CSV formatting.")
    exit()

print(f"✅ Data successfully cleaned! Shape: {data.shape}")

# ---------------------------
# 🔹 Feature Scaling (MinMax Scaling)
# ---------------------------
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(data.values)

# ---------------------------
# 🔹 Train-Test Split
# ---------------------------
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
input_dim = X_train.shape[1]  # Number of categories

print(f"📊 Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")

# ---------------------------
# 🔹 Build the Autoencoder Model
# ---------------------------
inp = Input(shape=(input_dim,))
encoded = Dense(int(input_dim / 2), activation='relu')(inp)
decoded = Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = Model(inp, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# ---------------------------
# 🔹 Train the Model
# ---------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\n🚀 Training the Autoencoder...")
autoencoder.fit(
    X_train, X_train,
    epochs=5,
    batch_size=256,
    shuffle=True,
    validation_data=(X_test, X_test),
    callbacks=[early_stop]
)

# ---------------------------
# 🔹 Save Model & Scaler
# ---------------------------
autoencoder.save("autoencoder.h5")
joblib.dump(scaler, "scaler.save")

print("\n✅ Model and scaler saved successfully!")


✅ Dataset loaded successfully!

📊 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458179 entries, 0 to 458178
Data columns (total 30 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   user_id                458179 non-null  object 
 1   resorts                458179 non-null  float64
 2   burger/pizza shops     458179 non-null  float64
 3   hotels/other lodgings  458179 non-null  float64
 4   juice bars             458179 non-null  float64
 5   beauty & spas          458179 non-null  float64
 6   gardens                458179 non-null  float64
 7   Amusement Parks        458179 non-null  float64
 8   Farmer market          458179 non-null  float64
 9   Market                 458179 non-null  float64
 10  Music halls            458179 non-null  float64
 11  Nature                 458179 non-null  float64
 12  Tourist attractions    458179 non-null  float64
 13  beaches                458179 non-null  f


✅ Model and scaler saved successfully!


In [3]:
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError

# Define the custom loss function explicitly
custom_objects = {"mse": MeanSquaredError()}

# Load the trained model and scaler
autoencoder = load_model("autoencoder.h5", custom_objects=custom_objects)
scaler = joblib.load("scaler.save")

print("✅ Model and scaler loaded successfully!")


✅ Model and scaler loaded successfully!


In [4]:
# Load new user data (replace this with real user data)
new_user_data = pd.DataFrame([{
    "resorts": 4.0,
    "burger/pizza shops": 3.5,
    "hotels/other lodgings": 4.5,
    "juice bars": 2.0,
    "beauty & spas": 3.0,
    "gardens": 4.2,
    "Amusement Parks": 1.5,
    "Farmer market": 4.1,
    "Market": 3.0,
    "Music halls": 2.5,
    "Nature": 4.8,
    "Tourist attractions": 4.6,
    "beaches": 4.0,
    "parks": 3.8,
    "theatres": 3.2,
    "museums": 3.9,
    "malls": 4.2,
    "restaurants": 4.5,
    "pubs/bars": 3.0,
    "local services": 2.9,
    "art galleries": 3.7,
    "dance clubs": 3.1,
    "swimming pools": 4.3,
    "bakeries": 3.0,
    "cafes": 3.2,
    "view points": 4.4,
    "monuments": 4.1,
    "zoo": 3.8,
    "supermarket": 4.5
}])

# Scale the data using the same scaler
new_user_scaled = scaler.transform(new_user_data)

print("✅ New user data prepared for prediction!")


✅ New user data prepared for prediction!


/Users/alexandrakhreiche/miniforge3/envs/tf_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [5]:
# Pass the scaled data through the autoencoder
predicted_preferences = autoencoder.predict(new_user_scaled)

# Convert back to original scale
predicted_preferences_original = scaler.inverse_transform(predicted_preferences)

# Convert results into a DataFrame
predicted_df = pd.DataFrame(predicted_preferences_original, columns=new_user_data.columns)

print("\n🔮 **Predicted User Preferences:**")
print(predicted_df)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step

🔮 **Predicted User Preferences:**
    resorts  burger/pizza shops  hotels/other lodgings  juice bars  \
0  2.692629            2.313426               2.290289    2.192776   

   beauty & spas   gardens  Amusement Parks  Farmer market    Market  \
0       1.976413  2.204842         2.519278       4.119593  3.940332   

   Music halls  ...  local services  art galleries  dance clubs  \
0     4.120941  ...        1.130583       3.263303     1.928581   

   swimming pools  bakeries     cafes  view points  monuments       zoo  \
0        1.739481    2.3837  3.403669     2.827838   4.066986  3.985428   

   supermarket  
0     4.055188  

[1 rows x 29 columns]
